2019 - Employment Permit by Nationality

In [1]:
year = 2019

# Reading the raw data in the folder on my personal computer
import pandas as pd

df = pd.read_excel(f"G:/My Drive/ESTUDOS DATA SCIENCE/ie-employment-permit/data/raw_data/{year}/permits-by-nationality-{year}.xlsx",
                   header=1,
                   skiprows=[2] # I needed to skip these rows because I had the summarization of the values by year, and for december which was uninteresting
)

In [2]:
# Replacing the NaN values by the year
df["Year"] = df["Year"].fillna(f"{year}")

In [3]:
# Before 2020 the column structures of the files were different. So, I needed to rename the column "Total" by "Issued" because after 2020 the new file didn't have the columns "New" and "Renewals", only "Issued". In this case, to keep the standard of the project, I needed to mantain only "Issued".
df.rename(columns={"Total": "Issued", "Nationality": "Country"}, inplace=True) # I had to replace "Nationality" by just "Country" once that this analysis is about countries
df.drop(columns=["New", "Renewal"], inplace=True) # Here I realized that the column "Sector" that was empty and it was useless

# I had to make some data cleaning before because some countries had different names years ago
df = df.loc[:, ~df.columns.str.contains('^Unnamed')] # The structure for this sheet was different, so I needed to remove an empty column

In [4]:
# I grouped by country because in some cases when I was fixing I replaced the old name to the new name, ie Hong Kong -> China making double China
df_grouped = df.groupby(["Year", "Country"], dropna=False).agg({
    "Issued": "sum",
    "Refused": "sum",
    "Withdrawn": "sum"
}).reset_index()

In [5]:
# Create the Primary Key
df_grouped["Year"] = df_grouped["Year"].astype(str)
df_grouped["Country"] = df_grouped["Country"].astype(str).str.strip()
df_grouped["id_country"] = df_grouped["Year"] + df_grouped["Country"].str.replace(r"\s+", "_", regex=True)

# Reorder columns
df_grouped = df_grouped[["id_country", "Year", "Country", "Issued", "Refused", "Withdrawn"]]

In [6]:
# I used the extension .csv because is lighter and easy to work with some libraries like pandas, sqlalchemy
df_grouped.to_csv(f"G:/My Drive/ESTUDOS DATA SCIENCE/ie-employment-permit/data/{year}/permits-by-nationality-{year}.csv", index=False)

print(df_grouped)

          id_country  Year      Country  Issued  Refused  Withdrawn
0    2019Afghanistan  2019  Afghanistan       6        0          0
1        2019Albania  2019      Albania      20        7          2
2        2019Algeria  2019      Algeria      10        1          2
3         2019Angola  2019       Angola       1        0          0
4      2019Argentina  2019    Argentina      61        8          2
..               ...   ...          ...     ...      ...        ...
110    2019Venezuela  2019    Venezuela      50       13          8
111      2019Vietnam  2019      Vietnam      98       13          3
112        2019Yemen  2019        Yemen       6        0          1
113       2019Zambia  2019       Zambia       6        1          0
114     2019Zimbabwe  2019     Zimbabwe     123        8          6

[115 rows x 6 columns]
